# Lesson 01 - Introducere în Agenții AI

Bine ați venit la prima lecție din cursul **Agenți AI pentru Începători**!

Un **agent AI** este un program care folosește un model de limbaj mare (LLM) ca motor de raționament și poate lua *acțiuni* în lumea reală — apelând API-uri, interogând baze de date sau rulând cod — pentru a îndeplini un scop în numele unui utilizator.

În acest caiet veți construi primul vostru agent: un **Agent de Călătorii** care recomandă destinații de vacanță. Pe parcurs veți învăța cum să:

1. Vă conectați la Azure AI Foundry Agent Service folosind **Microsoft Agent Framework**.
2. Oferiți agentului un **instrument** — o funcție simplă Python pe care o poate apela.
3. Rulați agentul și să inspectați răspunsul său.
4. Transmiteți în flux răspunsul agentului token cu token.


## Configurare

Înainte de a rula acest notebook, asigură-te că ai:

1. **Un proiect Azure AI Foundry** cu un model de chat implementat (de exemplu `gpt-4o-mini`).
2. **Conectat cu CLI-ul Azure** — rulează `az login` în terminalul tău.
3. **Setat variabilele de mediu necesare:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint-ul proiectului tău Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — numele modelului tău implementat.

Celula de mai jos instalează pachetele Python de care ai nevoie.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Crearea Primului Tău Agent

Un agent are nevoie de două lucruri:

- **Instrucțiuni** care îi spun *cine este* și *cum să se comporte* (un prompt de sistem).
- **Instrumente** — funcții Python decorate cu `@tool` pe care agentul le poate apela pentru a obține informații sau pentru a efectua acțiuni.

Mai jos definim un instrument simplu care returnează o listă de destinații populare de vacanță. Agentul va folosi acest instrument când un utilizator cere recomandări de călătorie.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Răspunsuri în Flux

Pentru o experiență mai interactivă, poți **transmite în flux** răspunsul agentului. În loc să aștepți răspunsul complet, agentul oferă bucăți de text pe măsură ce sunt generate. Acest lucru este deosebit de util în interfețele de chat unde dorești să afișezi ieșirea în timp real.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Rezumat

În această lecție ai învățat cum să:

- **Creezi un furnizor** care se conectează la Azure AI Foundry Agent Service prin `AzureAIProjectAgentProvider`.
- **Definești un instrument** folosind decoratorul `@tool` astfel încât agentul să poată apela funcțiile tale Python.
- **Rulezi agentul** cu un mesaj de la utilizator și să afișezi răspunsul acestuia.
- **Transmiți răspunsuri în flux** pentru output în timp real.

În următoarea lecție vom explora cadre agentice mai în detaliu și vom învăța cum să oferim agenților instrumente mai puternice și capacități de raționament în mai multi pași.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare de responsabilitate**:  
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automatizate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă o traducere profesională realizată de un traducător uman. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite ce pot apărea din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
